<a href="https://colab.research.google.com/github/shubhiv02-learner/banking-log-anomaly-detection/blob/main/notebooks/SentinelIq_gendata_TestModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# SentinelIQ V 0.2
# Statistical Observability Platform
# ============================================================

# This project simulates a banking observability pipeline
# capable of detecting infrastructure anomalies before
# they affect critical financial systems.
#
# Core Techniques Used:
# - EWMA Drift Detection
# - CUSUM Anomaly Detection
# - Persistence Scoring
# - Correlation Analysis
# - Bayesian Incident Prioritization
#
# Author: Shubhi Verma
# Environment: Google Colab / Jupyter Notebook
# ============================================================


# =========================
# STEP 1 — IMPORT LIBRARIES
# =========================

# Pandas -> Data manipulation
# NumPy -> Numerical operations
# PyPlot -> Dashboard visualization

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot
import seaborn as sns
import warnings
import os
import io

warnings.filterwarnings("ignore")
folders = [
    "src",
    "scripts",
    "data/raw",
    "data/processed",
    "outputs",
    "visualizations",
    "models",
    "reports",
    "notebooks",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project structure created.")

Project structure created.


In [6]:
%%writefile scripts/generate_banking_logs.py



import pandas as pd
import numpy as np
import random
import json
import os
from datetime import datetime, timedelta
import io # Added this import


# =========================
# CREATE OUTPUT DIRECTORY
# =========================

os.makedirs("outputs", exist_ok=True)


# ============================================================
# CONFIGURATION
# ============================================================

TOTAL_LOGS = 150000

START_TIME = datetime(2026, 1, 1)

SERVICES = [
    "payment-api",
    "auth-service",
    "trading-engine",
    "fraud-detection",
    "notification-service",
    "portfolio-service",
    "investment-engine"
]

ENDPOINTS = [
    "/login",
    "/transfer",
    "/payment",
    "/trade",
    "/portfolio",
    "/kyc",
    "/withdrawal"
]

# Map services to their specific endpoints
SERVICE_ENDPOINTS_MAP = {
    "payment-api": ["/payment", "/transfer", "/withdrawal"],
    "auth-service": ["/login", "/kyc"],
    "trading-engine": ["/trade", "/payment", "/transfer"],
    "fraud-detection": ["/kyc", "/payment", "/transfer"],
    "notification-service": ["/login", "/transfer", "/payment"],
    "portfolio-service": ["/portfolio", "/trade"],
    "investment-engine": ["/trade", "/portfolio", "/withdrawal"]
}

# Map services to their base latencies
SERVICE_LATENCY_MAP = {
    "payment-api": 150,
    "auth-service": 80,
    "trading-engine": 100,
    "fraud-detection": 200,
    "notification-service": 70,
    "portfolio-service": 120,
    "investment-engine": 130
}

REGIONS = [
    "India",
    "Singapore",
    "USA",
    "Germany",
    "UAE"
]

STATUS_TYPES = [
    "success",
    "timeout",
    "db_failure",
    "high_latency",
    "queue_delay"
]

# ANOMALY CONFIGURATION: Inject anomalies in three specific windows
ANOMALY_WINDOWS = [
    (int(TOTAL_LOGS * 0.1), int(TOTAL_LOGS * 0.12)), # 10-12% of logs
    (int(TOTAL_LOGS * 0.4), int(TOTAL_LOGS * 0.42)), # 40-42% of logs
    (int(TOTAL_LOGS * 0.7), int(TOTAL_LOGS * 0.72))  # 70-72% of logs
]
ANOMALY_PROB_IN_WINDOW = 0.8  # High probability of anomaly within a window
ANOMALY_PROB_OUT_WINDOW = 0.001 # Low probability of anomaly outside a window

# Define anomaly types and their weights to achieve the desired distribution
anomaly_types_for_selection = ["timeout", "db_failure", "high_latency", "queue_delay"]
anomaly_weights_for_selection = [0.03, 0.01, 0.4, 0.56] # 4% critical, 96% medium

# ============================================================
# GENERATE SINGLE LOG EVENT
# ============================================================

def generate_log_event(index):

    """
    Generates one structured banking log event.

    Simulates:
    - infrastructure metrics
    - service health
    - latency behavior
    - operational failures
    """

    # --------------------------------------------------------
    # Generate timestamp
    # --------------------------------------------------------

    timestamp = START_TIME + timedelta(
        seconds=index * 2
    )

    # --------------------------------------------------------
    # Randomly select banking service and its associated endpoint
    # --------------------------------------------------------

    service = random.choice(SERVICES)
    possible_endpoints = SERVICE_ENDPOINTS_MAP.get(service, ENDPOINTS) # Default to all endpoints if service not in map
    endpoint = random.choice(possible_endpoints)

    region = random.choice(REGIONS)

    # --------------------------------------------------------
    # Generate normal operational metrics with service-specific latency
    # --------------------------------------------------------

    base_latency = SERVICE_LATENCY_MAP.get(service, 120) # Default to 120 if service not in map
    latency_ms = int(
        np.random.normal(base_latency, 20) # Randomize around the base_latency
    )
    cpu_usage = round(np.random.normal(55, 8),2)
    memory_usage = round(np.random.normal(60, 10),2)
    queue_lag = max(0,int(np.random.normal(5, 2)))
    error_code = None
    status = "success"
    severity = "low"
    error_count = 0 # Initialize error_count
    is_anomaly = 0 # Initialize is_anomaly
    # --------------------------------------------------------
    # Inject anomalies probabilistically based on defined windows
    # --------------------------------------------------------

    in_anomaly_window = False
    for start, end in ANOMALY_WINDOWS:
        if start <= index <= end:
            in_anomaly_window = True
            break

    anomaly_roll = random.random()

    # Determine anomaly probability based on whether the current index is in an anomaly window
    current_anomaly_probability = ANOMALY_PROB_IN_WINDOW if in_anomaly_window else ANOMALY_PROB_OUT_WINDOW

    if anomaly_roll < current_anomaly_probability:
        is_anomaly = 1
        # Select anomaly type based on defined weights
        anomaly_type = random.choices(anomaly_types_for_selection, weights=anomaly_weights_for_selection, k=1)[0]

        # =========================================
        # TIMEOUT EVENT
        # =========================================

        if anomaly_type == "timeout":

            latency_ms =max(base_latency+50,int(np.random.normal(base_latency, 100))) # Randomize around the base_latency
            cpu_usage += random.randint(10, 30)
            queue_lag += random.randint(10, 40)
            error_code = "TIMEOUT_ERROR"
            status = "timeout"
            severity = "critical"
            error_count = random.randint(5, 10) # Assign a numerical error count

        # =========================================
        # DATABASE FAILURE
        # =========================================

        elif anomaly_type == "db_failure":

            latency_ms =max(base_latency+10,int(np.random.normal(base_latency, 80))) # Randomize around the base_latency
            cpu_usage += random.randint(20, 50)
            memory_usage += random.randint(10, 20)
            error_code = "DB_CONNECTION_FAILURE"
            status = "db_failure"
            severity = "critical"
            error_count = random.randint(5, 10) # Assign a numerical error count

        # =========================================
        # HIGH LATENCY EVENT
        # =========================================

        elif anomaly_type == "high_latency":
            latency_ms =max(base_latency+30,int(np.random.normal(base_latency, 80))) # Randomize around the base_latency
            queue_lag += random.randint(5, 15)
            error_code = "LATENCY_SPIKE"
            status = "high_latency"
            severity = "medium"
            error_count = random.randint(2, 5) # Assign a numerical error count

        # =========================================
        # QUEUE DELAY EVENT
        # =========================================

        elif anomaly_type == "queue_delay":

            latency_ms =max(base_latency+20,int(np.random.normal(base_latency, 50))) # Randomize around the base_latenc
            queue_lag += random.randint(20, 60)
            error_code = "QUEUE_BACKPRESSURE"
            status = "queue_delay"
            severity = "medium"
            error_count = random.randint(2, 5) # Assign a numerical error count

    # --------------------------------------------------------
    # Create structured log event
    # --------------------------------------------------------
    latency_ms = abs(latency_ms)

    log_event = {

        "timestamp": timestamp.isoformat(),
        "service": service,
        "endpoint": endpoint,
        "region": region,
        "latency_ms": latency_ms,
        "cpu_usage": round(cpu_usage, 2),
        "memory_usage": round(memory_usage, 2),
        "queue_lag": queue_lag,
        "status": status,
        "error_code": error_code,
        "severity": severity,
        "error_count": error_count, # Add error_count to the log event
         "is_anomaly": is_anomaly # Add is_anamoly to the log event
    }

    return log_event


# ============================================================
# GENERATE COMPLETE DATASET
# ============================================================

def generate_banking_logs(total_logs):

    """
    Generates complete banking observability dataset.
    """

    logs = []

    for i in range(total_logs):

        log = generate_log_event(i)

        logs.append(log)

    return logs


# ============================================================
# SAVE LOGS
# ============================================================

def save_logs(logs):

    """
    Saves logs in:
    - JSON format
    - CSV format
    """

    # --------------------------------------------------------
    # Save JSON logs
    # --------------------------------------------------------

    with open(
        "data/banking_logs.json",
        "w"
    ) as f:

        json.dump(
            logs,
            f,
            indent=4
        )

    # --------------------------------------------------------
    # Save CSV logs
    # --------------------------------------------------------

    df = pd.DataFrame(logs)

    df.to_csv(
        "data/banking_logs.csv",
        index=False
    )

    # Capture the statistics output string
    stats_string = display_statistics(df)

    # Define the output file path
    output_file_path = 'outputs/logsStatistics.txt'

    # Ensure the 'outputs' directory exists
    os.makedirs(os.path.dirname(output_file_path), exist_ok=True)

    # Write the captured output to the file
    with open(output_file_path, 'w') as f:
        f.write(stats_string)

    print(f"Log statistics saved to '{output_file_path}'") # Print confirmation here

    return df


# ============================================================
# DISPLAY SAMPLE STATISTICS
# ============================================================

def display_statistics(df):

    """
    Displays dataset overview and anomaly distribution.
    Returns the statistics as a string.
    """
    output_string = io.StringIO()

    output_string.write("\n================================================")
    output_string.write("\n SENTINELIQ LOG GENERATION COMPLETED ")
    output_string.write("\n================================================\n")
    output_string.write(f"\nTotal Logs Generated : {len(df)}\n")
    output_string.write("\nServices Monitored:")
    output_string.write(df['service'].value_counts().to_string())
    output_string.write("\n\nStatus Distribution:")
    output_string.write(df['status'].value_counts().to_string())
    output_string.write("\n\nSeverity Distribution:")
    output_string.write(df['severity'].value_counts().to_string())
    output_string.write("\n\nSample Logs:")
    output_string.write(df.head().to_string())

    # Print to stdout as well for immediate display
    print(output_string.getvalue())
    print("printed sample logs")
    return output_string.getvalue()


# ============================================================
# MAIN DATA GENERATION EXECUTION PIPELINE
# ============================================================

def main():

    print("\nGenerating banking observability logs...\n")

    logs = generate_banking_logs(TOTAL_LOGS)

    df = save_logs(logs)


    print("\nFiles Saved:\n")
    print(f"Banking logs in json saved to outputs/banking_logs.json ")
    print(f"Banking logs in csv format saved to outputs/banking_logs.csv ")
    print(f"Log statistics saved to outputs/logsStatistics.txt")
    print("\nLog generation completed successfully.\n")
    return df

Overwriting scripts/generate_banking_logs.py


In [7]:
%%writefile scripts/call_generate_banking_logs.py
import generate_banking_logs

if __name__ == "__main__":
    df = generate_banking_logs.main()
    print(df.head())


Writing scripts/call_generate_banking_logs.py


In [8]:
! python scripts/call_generate_banking_logs.py


Generating banking observability logs...


 SENTINELIQ LOG GENERATION COMPLETED 

Total Logs Generated : 150000

Services Monitored:service
investment-engine       21629
notification-service    21594
payment-api             21491
auth-service            21488
trading-engine          21407
fraud-detection         21245
portfolio-service       21146

Status Distribution:status
success         142638
queue_delay       4073
high_latency      2955
timeout            247
db_failure          87

Severity Distribution:severity
low         142638
medium        7028
critical       334

Sample Logs:             timestamp               service    endpoint     region  latency_ms  cpu_usage  memory_usage  queue_lag   status error_code severity  error_count  is_anomaly
0  2026-01-01T00:00:00  notification-service      /login    Germany          47      70.95         60.06          8  success       None      low            0           0
1  2026-01-01T00:00:02  notification-service    /payment        

### Load the Isolation Forest Model and Banking Logs Data

First, we'll load the pre-trained `Isolation Forest` model using `joblib` and the `banking_logs.csv` file into a pandas DataFrame.

In [13]:
import joblib
import pandas as pd

# Define paths
model_path = 'models/Isolation_forest_RawData.joblib'
data_path = 'data/banking_logs.csv'

# Load the Isolation Forest model
try:
    isolation_forest_model = joblib.load(model_path)
    print(f"Successfully loaded Isolation Forest model from {model_path}")
except FileNotFoundError:
    print(f"Error: Model file not found at {model_path}")
    isolation_forest_model = None

# Load the banking logs data
try:
    banking_df = pd.read_csv(data_path)
    print(f"Successfully loaded banking logs data from {data_path}")
    display(banking_df.head())
except FileNotFoundError:
    print(f"Error: Data file not found at {data_path}")
    banking_df = None

Successfully loaded Isolation Forest model from models/Isolation_forest_RawData.joblib
Successfully loaded banking logs data from data/banking_logs.csv


,timestamp,service,endpoint,region,latency_ms,cpu_usage,memory_usage,queue_lag,status,error_code,severity,error_count,is_anomaly
0,2026-01-01T00:00:00,notification-service,/login,Germany,47,70.95,60.06,8,success,NaN,low,0,0
1,2026-01-01T00:00:02,notification-service,/payment,USA,73,42.84,59.63,5,success,NaN,low,0,0
2,2026-01-01T00:00:04,portfolio-service,/portfolio,UAE,107,55.65,64.63,4,success,NaN,low,0,0
3,2026-01-01T00:00:06,investment-engine,/trade,Singapore,118,52.18,63.65,4,success,NaN,low,0,0
4,2026-01-01T00:00:08,investment-engine,/portfolio,India,127,55.35,55.04,6,success,NaN,low,0,0


In [16]:
if isolation_forest_model is not None:
    print("\nFeatures the model was trained with (feature_names_in_):")
    if hasattr(isolation_forest_model, 'feature_names_in_'):
        print(isolation_forest_model.feature_names_in_)
    else:
        print("The loaded model does not have a 'feature_names_in_' attribute. This might be an older sklearn version or a custom model.")
else:
    print("Model not loaded, cannot check feature names.")


Features the model was trained with (feature_names_in_):
['service' 'latency_ms' 'cpu_usage' 'queue_lag' 'error_count']


### Prepare Data and Predict Anomalies

Next, we need to prepare the `banking_logs` data by selecting the numerical features that the `Isolation Forest` model expects. Then, we will use the loaded model to predict anomalies on this prepared data.

In [17]:
import joblib
import pandas as pd
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
from sklearn.preprocessing import LabelEncoder # Import LabelEncoder

if isolation_forest_model is not None and banking_df is not None:
    # Get the feature names the model was trained with
    if hasattr(isolation_forest_model, 'feature_names_in_'):
        model_feature_names = list(isolation_forest_model.feature_names_in_)
        print(f"Model expects features: {model_feature_names}")
    else:
        print("Error: Model does not have 'feature_names_in_' attribute. Cannot proceed safely.")
        # Handle the error, perhaps exit or raise an exception
        model_feature_names = [] # Set to empty to prevent further errors

    if not model_feature_names:
        print("Model feature names could not be determined, skipping prediction.")
    else:
        # Make a copy of the banking_df containing only the features expected by the model
        data_for_prediction = banking_df[model_feature_names].copy()

        # Identify categorical and numerical columns from the model's expected features
        # Based on previous trial and error, 'service' is categorical and others are numerical.
        categorical_cols = ['service'] if 'service' in model_feature_names else []
        numerical_cols = [col for col in model_feature_names if col not in categorical_cols]

        # Handle potential non-numeric values and NaNs in numerical columns
        for col in numerical_cols:
            data_for_prediction[col] = pd.to_numeric(data_for_prediction[col], errors='coerce')
            # Fill NaNs using the median of each column
            data_for_prediction[col] = data_for_prediction[col].fillna(data_for_prediction[col].median())

        # Apply LabelEncoder to categorical 'service' column if present
        if 'service' in categorical_cols:
            le = LabelEncoder()
            data_for_prediction['service'] = le.fit_transform(data_for_prediction['service'])

        # Ensure the order of columns in X_test matches model_feature_names
        X_test = data_for_prediction[model_feature_names]

        print("Shape of data prepared for prediction:", X_test.shape)
        display(X_test.head())

        # Predict anomalies
        banking_df['anomaly_score'] = isolation_forest_model.decision_function(X_test)
        banking_df['anomaly_prediction'] = isolation_forest_model.predict(X_test)

        # Display the first few rows with anomaly scores and predictions
        print("Banking logs with anomaly scores and predictions:")
        display(banking_df[['timestamp', 'service', 'status', 'is_anomaly', 'anomaly_score', 'anomaly_prediction']].head(10))

        # Count predicted anomalies
        num_predicted_anomalies = banking_df[banking_df['anomaly_prediction'] == -1].shape[0]
        print(f"Number of predicted anomalies: {num_predicted_anomalies}")

        # Count actual anomalies (if 'is_anomaly' column is present and indicates anomalies with 1)
        if 'is_anomaly' in banking_df.columns:
            num_actual_anomalies = banking_df[banking_df['is_anomaly'] == 1].shape[0]
            print(f"Number of actual anomalies: {num_actual_anomalies}")

            # Convert ground truth anomalies (1) to -1 for consistency with Isolation Forest prediction style
            # and non-anomalies (0) to 1.
            y_true = banking_df['is_anomaly'].apply(lambda x: -1 if x == 1 else 1)
            y_pred = banking_df['anomaly_prediction']

            print("\nConfusion Matrix (Actual vs. Predicted):")
            print(confusion_matrix(y_true, y_pred))

            print("\nClassification Report:")
            print(classification_report(y_true, y_pred))

else:
    print("Model or data not loaded, skipping anomaly prediction.")

Model expects features: ['service', 'latency_ms', 'cpu_usage', 'queue_lag', 'error_count']
Shape of data prepared for prediction: (150000, 5)


,service,latency_ms,cpu_usage,queue_lag,error_count
0,3,47,70.95,8,0
1,3,73,42.84,5,0
2,5,107,55.65,4,0
3,2,118,52.18,4,0
4,2,127,55.35,6,0


Banking logs with anomaly scores and predictions:


,timestamp,service,status,is_anomaly,anomaly_score,anomaly_prediction
0,2026-01-01T00:00:00,notification-service,success,0,0.033328,1
1,2026-01-01T00:00:02,notification-service,success,0,0.132903,1
2,2026-01-01T00:00:04,portfolio-service,success,0,0.166853,1
3,2026-01-01T00:00:06,investment-engine,success,0,0.164063,1
4,2026-01-01T00:00:08,investment-engine,success,0,0.166964,1
5,2026-01-01T00:00:10,auth-service,success,0,0.134756,1
6,2026-01-01T00:00:12,portfolio-service,success,0,0.085121,1
7,2026-01-01T00:00:14,payment-api,success,0,0.067357,1
8,2026-01-01T00:00:16,auth-service,success,0,0.113338,1
9,2026-01-01T00:00:18,portfolio-service,success,0,0.145920,1


Number of predicted anomalies: 7575
Number of actual anomalies: 7362

Confusion Matrix (Actual vs. Predicted):
[[  7272     90]
 [   303 142335]]

Classification Report:
              precision    recall  f1-score   support

          -1       0.96      0.99      0.97      7362
           1       1.00      1.00      1.00    142638

    accuracy                           1.00    150000
   macro avg       0.98      0.99      0.99    150000
weighted avg       1.00      1.00      1.00    150000

